# Article text representations

**Input:** prepared article catalogue. **Output:** title64, abstract64, text64, title tokens and category IDs.
Fit vocabulary, IDF and SVD on training titles only; freeze them for validation/test articles.
Row 0 is padding. Nonzero float32 vectors have unit length; missing vectors remain zero.
The 0.7 title / 0.3 abstract mixture is a baseline setting, not a validated optimum.

Run the cells in order. Code must be available in this runtime, including `src/newsrec`.
For Colab, upload the full checkout or mount its folder on Drive; opening only the notebook does not upload the package.
Set `NEWSREC_CODE` before the setup cell if the checkout is elsewhere. Set `DATA_ROOT` for separate datasets.
Inputs are prepared parquet files under `1_Data_Preparation/processed`; outputs go to `3_Representations/artifacts`.
The calculation cells below are passed to shared storage helpers.

In [ ]:
from pathlib import Path
import os
import sys
import json
import numpy as np
import pandas as pd
from IPython.display import display

# Code: run from the Group 6 checkout, or set NEWSREC_CODE to that folder.
CODE_ROOT = Path(os.environ['NEWSREC_CODE']) if 'NEWSREC_CODE' in os.environ else next(
    p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'src/newsrec').is_dir()
)
sys.path.insert(0, str(CODE_ROOT / 'src'))

# Data: use the checkout by default. For Drive, mount it first and set DATA_ROOT.
DATA_ROOT = None  # Colab example: '/content/drive/MyDrive/BT4222'
ROOT = Path(DATA_ROOT or os.getenv('NEWSREC_ROOT') or CODE_ROOT).expanduser().resolve()
ARTIFACTS = ROOT / '3_Representations/artifacts'
print('Code:', CODE_ROOT)
print('Data and outputs:', ROOT)

from newsrec.common import Settings, saved_stage_ready

from newsrec.representations import build_news, representation_summary, text_neighbors
from collections import Counter
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from newsrec.common import DEFAULT_SETTINGS, normalise
from newsrec.representations import clean_text

In [ ]:
SETTINGS = Settings()  # Fixed seed and dimensions; use the same settings in all stages.
REBUILD = False  # Set True after editing a calculation cell. Shared code changes invalidate caches.

## 1. Fit and export

Defaults: title unigrams/bigrams, up to 50,000 terms, `min_df=2`, SVD64, fixed seed.
All vectors are float32 and L2-normalised when valid. The combined view weights title/abstract 0.7/0.3;
a missing/OOV field falls back to the other valid field. Completely unsupported text remains zero with a mask.
Array row 0 is reserved for padding; real `news_code` values must go through `article_index.parquet`.
Training-fitted category/subcategory vocabularies reserve code 0 for unseen values.

Rebuilding representations invalidates dependent artifacts. Use one settings across the three notebooks.

### Training-only text fit and frozen transforms

The mask comes from `in_news_train`. Only training titles enter `fit_transform` or `svd.fit`; all catalogue titles and abstracts use the fitted models through `transform`. Row 0 is padding; missing/unsupported content stays masked.

**Scope:** training-only fitting prevents validation/test fitting leakage. This is an offline training-catalogue representation, not a text model refitted at each training impression timestamp.

In [ ]:
def fit_text_space(news, cfg=DEFAULT_SETTINGS):
    """Fit vocabulary/IDF and SVD on training titles; freeze for every transform."""
    titles = news.title.map(clean_text)
    abstracts = news.abstract.map(clean_text)
    train = news.in_news_train.fillna(False).to_numpy(dtype=bool)
    if train.sum() < 2:
        raise ValueError("At least two training articles are required")
    # Learn vocabulary and IDF from training titles only.
    tfidf = TfidfVectorizer(
        ngram_range=(1, 2),
        max_features=cfg.max_features,
        min_df=cfg.min_df,
        dtype=np.float32,
    )
    train_tfidf = tfidf.fit_transform(titles[train])
    components = min(cfg.text_dim, train_tfidf.shape[0] - 1, train_tfidf.shape[1] - 1)
    if components < 1:
        raise ValueError(
            "Insufficient training vocabulary for SVD; inspect min_df/text"
        )
    svd = TruncatedSVD(n_components=components, random_state=cfg.seed)
    svd.fit(train_tfidf)
    # Transform both fields with the same frozen coordinates; reserve row 0.
    matrices = []
    for texts in (titles, abstracts):
        reduced = normalise(svd.transform(tfidf.transform(texts)))
        dense = np.zeros((len(titles) + 1, cfg.text_dim), np.float32)
        dense[1:, :components] = reduced
        matrices.append(dense)
    title, abstract = matrices
    tv = np.linalg.norm(title, axis=1) > 0
    av = np.linalg.norm(abstract, axis=1) > 0
    # Fuse valid fields; fall back to the available field when one is missing.
    text = normalise(cfg.title_weight * title + (1 - cfg.title_weight) * abstract)
    text[tv & ~av] = title[tv & ~av]
    text[av & ~tv] = abstract[av & ~tv]
    return tfidf, svd, components, title, abstract, text, tv, av

### Training vocabulary for NRMS and categories

Token/category vocabularies also use only training articles; unseen tokens get ID 1 and unseen categories ID 0.

In [ ]:
def encode_title_tokens(titles, train, cfg=DEFAULT_SETTINGS):
    tokenise = TfidfVectorizer().build_analyzer()
    frequency = Counter(token for text_ in titles[train] for token in tokenise(text_))
    token_vocab = {
        w: i + 2
        for i, (w, _) in enumerate(
            sorted(frequency.items(), key=lambda x: (-x[1], x[0]))[: cfg.max_features]
        )
    }
    token_ids = np.zeros((len(titles) + 1, cfg.title_tokens), np.int32)
    token_lengths, unknown, total = ([], 0, 0)
    for i, text_ in enumerate(titles, 1):
        tokens = tokenise(text_)
        token_lengths.append(len(tokens))
        ids = [token_vocab.get(w, 1) for w in tokens]
        unknown += sum(x == 1 for x in ids)
        total += len(ids)
        token_ids[i, : min(len(ids), cfg.title_tokens)] = ids[: cfg.title_tokens]
    return token_ids, token_vocab, token_lengths, unknown, total


def fit_category_encodings(news, train):
    encodings = {}
    for col in ("category", "subcategory"):
        vocabulary = sorted(set(news.loc[train, col].dropna().astype(str)))
        lookup = {v: i + 1 for i, v in enumerate(vocabulary)}
        codes = news[col].map(lookup).fillna(0).astype(np.int32)
        encodings[col] = (codes, lookup)
    return encodings

In [ ]:
if REBUILD or not saved_stage_ready(ROOT, SETTINGS, "news"):
    coverage = build_news(
        ROOT,
        SETTINGS,
        text_encoder=fit_text_space,
        token_encoder=encode_title_tokens,
        category_encoder=fit_category_encodings,
    )
else:
    print("Loaded saved article text representations.")
    report = pd.read_csv(ROOT / "results/representation_coverage.csv")
    coverage = report[report.stage.eq("text")]
display(coverage)

## 2. Check dimensions, norms and quality

A present title can become a zero vector after train-vocabulary filtering. Abstract absence and zero-vector
rates are distinct. On a tiny fixture the effective SVD rank can be smaller than the configured dimension;
unused dimensions are padded with zeros and the effective rank is recorded.

In [ ]:
display(representation_summary(ROOT, SETTINGS))
index = pd.read_parquet(ARTIFACTS / "article_index.parquet")
display(
    index[
        [
            "news_code",
            "title",
            "has_abstract",
            "has_title_vector",
            "has_abstract_vector",
        ]
    ].head()
)
meta = json.loads((ARTIFACTS / "news_manifest.json").read_text())
display(
    {
        k: meta[k]
        for k in [
            "effective_svd_dim",
            "fit_scope",
            "token_oov_fraction",
            "title_truncation_fraction",
        ]
    }
)

## 3. Inspect text quality

Compare three deterministic query articles across title, abstract and fused views.
Inspect topic relevance manually: high cosine alone does not establish recommendation quality.
The table below also counts train-vocabulary filtering separately for each field and split.

In [ ]:
import pickle
from newsrec.audits import view_neighbors

# Input: saved frozen transform. Output: field/split coverage and OOV fractions.
with (ARTIFACTS / 'text_transform.pkl').open('rb') as file:
    transform = pickle.load(file)
tfidf, svd = transform['tfidf'], transform['svd']
analyze = tfidf.build_analyzer()
catalogue = pd.read_parquet(ROOT / '1_Data_Preparation/processed/news_catalogue.parquet')
index = index.merge(catalogue[['news_code', 'in_news_train', 'in_news_validation', 'in_news_test']],
                    on='news_code', validate='one_to_one')
quality_rows = []
for split in ('train', 'validation', 'test'):
    subset = index.loc[index[f'in_news_{split}'].fillna(False)]
    for field in ('title', 'abstract'):
        texts = subset[field].map(clean_text)
        total = missing = 0
        for text in texts:
            tokens = analyze(text)
            total += len(tokens)
            missing += sum(token not in tfidf.vocabulary_ for token in tokens)
        sparse = tfidf.transform(texts)
        quality_rows.append(dict(split=split, field=field, articles=len(texts),
                                 empty_text=int(texts.eq('').sum()),
                                 zero_tfidf=int((sparse.getnnz(axis=1) == 0).sum()),
                                 oov_fraction=missing / total if total else np.nan))
text_quality = pd.DataFrame(quality_rows)
text_quality.to_csv(ARTIFACTS / 'text_quality.csv', index=False)
display(text_quality)
print('Train-title SVD explained variance:', svd.explained_variance_ratio_.sum())

# Output: 3 queries x 3 views x up to 3 neighbours, without click labels.
neighbors = view_neighbors(ROOT, queries=3, count=3)
neighbors.to_csv(ARTIFACTS / 'text_neighbors.csv', index=False)
display(neighbors)
display(pd.read_csv(ROOT / 'results/representation_cost.csv').tail())

## Handoff

Consumers can load the saved arrays with NumPy and the article index with pandas after all three notebooks finish.
`title64` reproduces the report's multi-interest space; `text64` is a separate combined-content view.
NRMS receives `title_token_ids.npy`, its mask, and the training-only vocabulary (`0=PAD`, `1=UNK`);
its attention encoders and learned embeddings belong to the NRMS notebook.